### Solução baseada no paper do link: https://ieeexplore.ieee.org/abstract/document/10896447
### https://github.com/ariandra34/Prepared-for-Lift-off/blob/main/main_solution.ipynb
### Prepared for Lift-Off: Hybrid CNN-LSTM Architecture for Aircraft Engine Remaining Useful Life Estimation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error,mean_absolute_error
from keras.models import Sequential
from keras.optimizers import Adam
from keras.layers import LSTM, Dense, Dropout, Conv1D, MaxPooling1D, Activation
from keras.callbacks import EarlyStopping

In [6]:
#Carregando os Dados, testando primeiro para o FD001

motor = 'FD001'
X_train = np.load(f"../data/processed_data/X_train_{motor}.npz")['dados']
y_train = np.load(f"../data/processed_data/y_train_{motor}.npz")['dados']
X_test = np.load(f"../data/processed_data/X_test_{motor}.npz")['dados']
y_test = np.load(f"../data/processed_data/y_test_{motor}.npz")['dados']

print(f"Data Shape da CNN: {X_train.shape[1:]} (Ciclos, Sensores)")

Data Shape da CNN: (30, 19) (Ciclos, Sensores)


In [7]:
model = Sequential()
model.add(Conv1D(64, 3, activation='relu', padding='same' , input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(MaxPooling1D(pool_size=1))
model.add(Dropout(0.1))
model.add(Conv1D(32 , 3, activation='relu', padding='same'))
model.add(MaxPooling1D(pool_size=1))
model.add(LSTM(100, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=1))
model.add(Activation("linear"))
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
print(model.summary())

C:\Users\davis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 30, 64)         │         3,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 30, 32)         │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 30, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 30, 100)        │        53,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 30, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 50)             │        30,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            51 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,339 (364.61 KB)

 Trainable params: 93,339 (364.61 KB)

 Non-trainable params: 0 (0.00 B)

None


In [8]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(X_train, y_train, batch_size=64, epochs=70, validation_split=0.2, callbacks=[early_stop], verbose=1)

#Avaliando no Teste
print("\nAvaliando o modelo nos dados de teste...")
y_pred = model.predict(X_test)

#Calculando as métricas
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"\nTreinamento Concluído!")
print(f"RMSE da CNN-LSTM no FD001: {rmse:.2f} ciclos")
print(f"MAE da CNN-LSTM no FD001: {mae:.2f} ciclos")

Epoch 1/70
222/222 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - loss: 6032.6592 - mae: 66.7861 - val_loss: 5795.1880 - val_mae: 65.8020
Epoch 2/70
222/222 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - loss: 4645.5181 - mae: 57.9122 - val_loss: 4605.3828 - val_mae: 58.4771
Epoch 3/70
222/222 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 3699.7078 - mae: 51.6826 - val_loss: 3720.4351 - val_mae: 52.8592
Epoch 4/70
222/222 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 3013.2898 - mae: 47.0670 - val_loss: 3074.6003 - val_mae: 48.5828
Epoch 5/70
222/222 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 2538.9412 - mae: 43.7313 - val_loss: 2618.9910 - val_mae: 45.4022
Epoch 6/70
222/222 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 2203.7031 - mae: 41.1938 - val_loss: 2297.8633 - val_mae: 42.4832
Epoch 7/70
222/222 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 1522.5288 - mae: 31.5534 - val_loss: 1525.8252 - val_mae: 32.3954
Epoch 8/70
222/222 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 1137.0322 - mae: 26.7027 - val_loss: 1123.6311 - v